# TerraScope — Satellite Change Intelligence Platform
## Research-Grade Multi-Architecture GPU Training & Benchmarking Pipeline

This notebook orchestrates the reproducible training and quantitative benchmarking of three bi-temporal change detection architectures on Google Colab:
1. **Siamese U-Net**: Convolutional baseline with spatial skip connections and difference feature fusion.
2. **Siamese Vision Transformer (ViT)**: Pure self-attention token-based architecture.
3. **Siamese Swin Transformer**: Shifted-window hierarchical vision transformer with multi-scale difference fusion.

### Benchmark Datasets:
- **LEVIR-CD**: Ultra-high-resolution bi-temporal optical building change detection (RGB).
- **OSCD**: Onera Satellite Change Detection multi-spectral Sentinel-2 dataset.

---
### Reliability, Provenance & Resilience Protocols:
- **Periodic Drive Checkpoint Backup**: Checkpoints (`latest.pt` and `best.pt`) are mirrored to Google Drive **at the end of every epoch**, protecting against Colab session timeouts.
- **Strict Stale-Checkpoint Protection**: When `FORCE_RETRAIN = True`, pre-existing checkpoints are safely segregated into an `archived/` subfolder. If a new run fails, stale weights from previous sessions can never be mistaken for current outputs.
- **Cryptographic Provenance in Manifest**: The experiment manifest records `run_id`, `status` (`RUNNING`, `COMPLETED`, `FAILED`), checkpoint path, and the checkpoint's internal `saved_at` timestamp.
- **Pre-Evaluation Provenance Gate**: Before benchmarking, the evaluation cell validates that every required model has a verified, newly completed checkpoint. Evaluation will **never** silently execute on unverified or stale files.
- **Non-Destructive Dataset Staging**: Optional configurable staging copies datasets to local fast SSD (`/content/data`) to maximize dataloader throughput while keeping original Google Drive files 100% untouched.
- **Empirical Batch-Size Guidance**: Batch size 8 is treated as an empirical starting baseline, not an absolute guarantee, with documented fallback to 4 or 2 for memory pressure.


### Step 1: Mount Google Drive & Setup Persistent Storage
Mount Google Drive and create persistent folders for checkpoints, archives, experiment logs, and visual analytics.


In [ ]:
from google.colab import drive
import os
import sys
import shutil
from pathlib import Path

drive.mount("/content/drive")

# Persistent storage directories on Google Drive
DRIVE_OUTPUT_DIR = Path("/content/drive/MyDrive/TerraScope_Outputs")
DRIVE_CHECKPOINTS_DIR = DRIVE_OUTPUT_DIR / "checkpoints"
DRIVE_ARCHIVE_DIR = DRIVE_CHECKPOINTS_DIR / "archived"
DRIVE_EXPERIMENTS_DIR = DRIVE_OUTPUT_DIR / "experiments"
DRIVE_VIZ_DIR = DRIVE_OUTPUT_DIR / "visualizations"

for p in (DRIVE_CHECKPOINTS_DIR, DRIVE_ARCHIVE_DIR, DRIVE_EXPERIMENTS_DIR, DRIVE_VIZ_DIR):
    p.mkdir(parents=True, exist_ok=True)

print(f" Persistent Google Drive storage verified at: {DRIVE_OUTPUT_DIR}")


### Step 2: GPU Accelerator & Memory Audit
Inspects GPU hardware and provides empirical, non-absolute memory guidance for batch size tuning.


In [ ]:
!nvidia-smi

import torch

if not torch.cuda.is_available():
    raise SystemError(
        "CRITICAL: CUDA GPU is not available! \n"
        "Go to Runtime > Change runtime type > Hardware accelerator > select T4 GPU (or A100/V100)."
    )

gpu_name = torch.cuda.get_device_name(0)
total_vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
print("--- GPU Accelerator Hardware Audit ---")
print(f" PyTorch Version : {torch.__version__}")
print(f" CUDA Device     : {gpu_name}")
print(f" Total VRAM      : {total_vram_gb:.2f} GB")

print("\nEmpirical Memory & Batch-Size Guidance:")
print("• Starting Baseline: Batch size 8 is an empirical starting point for 256x256 crops on 14-16 GB GPUs (T4).")
print("• Transformer Complexity: Transformer self-attention scales with token sequence length and model depth.")
print("  Batch size 8 is NOT an absolute guarantee across all workloads. Swin and ViT may consume more memory")
print("  during backpropagation than standard CNNs.")
print("• Out of Memory (OOM) Recovery: If any experiment encounters torch.cuda.OutOfMemoryError:")
print("  1. In the EXPERIMENT_MATRIX below, change batch_size from 8 to 4.")
print("  2. For Swin Transformer (hierarchical shifted window attention), reduce to 2 if necessary.")


### Step 3: Setup Project Workspace
Copies repository code to local fast SSD (`/content/satellite`) for optimal disk throughput while keeping persistent outputs linked to Google Drive.


In [ ]:
DRIVE_PROJECT_PATH = Path("/content/drive/MyDrive/satellite")
LOCAL_PROJECT_PATH = Path("/content/satellite")

if DRIVE_PROJECT_PATH.exists():
    if not LOCAL_PROJECT_PATH.exists():
        print(f"Copying project from {DRIVE_PROJECT_PATH} to {LOCAL_PROJECT_PATH} for SSD throughput...")
        shutil.copytree(str(DRIVE_PROJECT_PATH), str(LOCAL_PROJECT_PATH), dirs_exist_ok=True)
    os.chdir(str(LOCAL_PROJECT_PATH))
    PROJECT_ROOT = LOCAL_PROJECT_PATH
elif LOCAL_PROJECT_PATH.exists():
    os.chdir(str(LOCAL_PROJECT_PATH))
    PROJECT_ROOT = LOCAL_PROJECT_PATH
else:
    PROJECT_ROOT = Path(os.getcwd())

print(f" Active Project Root: {PROJECT_ROOT}")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))


### Step 4: Install Dependencies & Verify Core Modules
Installs required geospatial and deep learning packages (`rasterio`, `geopandas`, `timm`, `reportlab`).


In [ ]:
!pip install -q rasterio geopandas timm reportlab python-dotenv

# Verify crucial imports
import rasterio
import timm
from ml.models import get_model
from ml.utils.device import detect_device

print(f" Active Compute Device: {detect_device()}")
print(" Core dependencies verified successfully.")


### Step 5: Dataset Path Configuration & Structural Integrity Gate
Validates dataset directories before training.
For **OSCD**, points to the `onera DATA` root folder containing:
- `Onera Satellite Change Detection dataset - Images`
- `Onera Satellite Change Detection dataset - Train Labels`
- `Onera Satellite Change Detection dataset - Test Labels`

Execution halts with an informative error if required folders or splits are missing.


In [ ]:
# --- CONFIGURE YOUR DATASET PATHS ON GOOGLE DRIVE ---
LEVIR_CD_DRIVE_DIR = Path("/content/drive/MyDrive/data/LEVIR-CD")
OSCD_DRIVE_DIR = Path("/content/drive/MyDrive/data/onera DATA")

def validate_datasets(levir_path: Path, oscd_path: Path) -> dict:
    status = {"levir_ok": False, "oscd_ok": False}
    
    # 1. Validate LEVIR-CD
    if not levir_path.exists():
        print(f" Note: LEVIR-CD path not found at: {levir_path}")
    else:
        missing_levir = []
        for sp in ("train", "val", "test"):
            for sub in ("A", "B", "label"):
                d = levir_path / sp / sub
                if not d.is_dir() or len(list(d.glob("*.*"))) == 0:
                    missing_levir.append(str(d))
        if missing_levir:
            raise FileNotFoundError(
                f"LEVIR-CD missing required subdirectories or image tiles:\n" + "\n".join(missing_levir)
            )
        status["levir_ok"] = True
        print(f" LEVIR-CD validated successfully: {levir_path}")

    # 2. Validate OSCD
    if not oscd_path.exists():
        print(f" Note: OSCD path not found at: {oscd_path}")
    else:
        from ml.datasets.oscd import _find_images_base, _find_label_roots
        img_base = _find_images_base(oscd_path)
        train_lbls = _find_label_roots(oscd_path, explicit_dir=None, category="train")
        test_lbls = _find_label_roots(oscd_path, explicit_dir=None, category="test")
        
        if not img_base.is_dir():
            raise FileNotFoundError(f"OSCD Images folder not found under {oscd_path}")
        if not train_lbls:
            raise FileNotFoundError(f"OSCD Train Labels folder not found under {oscd_path}")
        if not test_lbls:
            raise FileNotFoundError(f"OSCD Test Labels folder not found under {oscd_path}")
        
        status["oscd_ok"] = True
        print(f" OSCD validated successfully:")
        print(f"   Root Dir    : {oscd_path.name}")
        print(f"   Images Base : {img_base.name}")
        print(f"   Train Labels: {train_lbls[0].name}")
        print(f"   Test Labels : {test_lbls[0].name}")

    if not status["levir_ok"] and not status["oscd_ok"]:
        raise SystemExit("CRITICAL: Neither LEVIR-CD nor OSCD was found at the configured paths. Stopping.")
    return status

dataset_status = validate_datasets(LEVIR_CD_DRIVE_DIR, OSCD_DRIVE_DIR)


### Step 6: Configurable Dataset Staging (Google Drive $\to$ Local Colab SSD)
Reading tens of thousands of image patches directly over Google Drive FUSE can cause I/O latency bottlenecks.
Setting `STAGE_DATASETS_TO_SSD = True` copies datasets to `/content/data/` for 5-10x faster dataloader performance.
Original Google Drive files remain completely untouched.


In [ ]:
STAGE_DATASETS_TO_SSD = True  # Set to False to stream directly from Drive if VM disk space is constrained

LOCAL_DATA_DIR = Path("/content/data")
LOCAL_DATA_DIR.mkdir(parents=True, exist_ok=True)

if STAGE_DATASETS_TO_SSD:
    print("=== Configurable Dataset Staging to Local SSD ===")
    
    # 1. Stage LEVIR-CD
    if dataset_status["levir_ok"]:
        local_levir = LOCAL_DATA_DIR / "LEVIR-CD"
        if not local_levir.exists():
            print(f"Staging LEVIR-CD from Drive to {local_levir} ...")
            shutil.copytree(str(LEVIR_CD_DRIVE_DIR), str(local_levir), dirs_exist_ok=True)
            print(" LEVIR-CD staged to local SSD.")
        else:
            print(" LEVIR-CD already staged on local SSD.")
        ACTIVE_LEVIR_DIR = local_levir
    else:
        ACTIVE_LEVIR_DIR = LEVIR_CD_DRIVE_DIR

    # 2. Stage OSCD
    if dataset_status["oscd_ok"]:
        local_oscd = LOCAL_DATA_DIR / "onera DATA"
        if not local_oscd.exists():
            print(f"Staging OSCD from Drive to {local_oscd} ...")
            shutil.copytree(str(OSCD_DRIVE_DIR), str(local_oscd), dirs_exist_ok=True)
            print(" OSCD staged to local SSD.")
        else:
            print(" OSCD already staged on local SSD.")
        ACTIVE_OSCD_DIR = local_oscd
    else:
        ACTIVE_OSCD_DIR = OSCD_DRIVE_DIR
else:
    print("Direct Streaming Mode: Reading datasets directly from Google Drive.")
    ACTIVE_LEVIR_DIR = LEVIR_CD_DRIVE_DIR
    ACTIVE_OSCD_DIR = OSCD_DRIVE_DIR

print(f"\nActive LEVIR-CD Path : {ACTIVE_LEVIR_DIR}")
print(f"Active OSCD Path     : {ACTIVE_OSCD_DIR}")


### Step 7: Pre-Flight GPU Smoke Test & Real-DataLoader Validation Gate
Performs two rapid validation gates:
- **Part A (In-Memory GPU Mechanics)**: Verifies CUDA forward pass, BCE+Dice loss calculation, backward gradients, optimizer step, and checkpoint save/reload integrity.
- **Part B (Real-DataLoader Integration)**: Fetches a single mini-batch from the real LEVIR-CD and OSCD datasets and verifies that all three architectures process real satellite input dimensions and channels without errors.

Execution stops immediately if any check fails.


In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from ml.models import get_model
from ml.training.losses import get_loss_function
from ml.training.optimizer import create_optimizer
from ml.training.callbacks import ModelCheckpointManager, load_checkpoint
from ml.datasets import LEVIRCDDataset, OSCDDataset

print("=== GATELIGHT 1: IN-MEMORY GPU MECHANICS SMOKE TEST ===")
device = torch.device("cuda")

try:
    # 1. Forward pass
    smoke_model = get_model("siamese_unet", in_channels=3, img_size=256, pretrained=False).to(device)
    smoke_model.train()
    t1 = torch.randn(2, 3, 256, 256, device=device)
    t2 = torch.randn(2, 3, 256, 256, device=device)
    target = torch.randint(0, 2, (2, 256, 256), device=device, dtype=torch.float32)
    
    logits = smoke_model(t1, t2)
    assert logits.shape == (2, 1, 256, 256), f"Unexpected output shape: {logits.shape}"
    
    # 2. Loss & Backward
    criterion = get_loss_function("bce_dice")
    loss = criterion(logits, target)
    assert torch.isfinite(loss), f"Loss is not finite: {loss.item()}"
    
    optimizer = create_optimizer(smoke_model, name="adamw", lr=3e-4)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    
    # 3. Checkpoint save & reload
    temp_ckpt_dir = Path("/tmp/smoke_checkpoints")
    ckpt_mgr = ModelCheckpointManager(temp_ckpt_dir, "smoke_unet", "test_ds")
    saved_path = ckpt_mgr.save(smoke_model, optimizer, None, epoch=1, best_metric=0.85, config={}, is_best=True)
    assert saved_path.is_file(), "Smoke checkpoint creation failed!"
    
    reload_meta = load_checkpoint(saved_path, smoke_model, optimizer, map_location=device)
    assert reload_meta.get("epoch") == 1, "Checkpoint reload failed!"
    print(" In-memory GPU mechanics: PASSED (Forward, Loss, Backward, Optimizer, Checkpoint).")
except Exception as e:
    raise SystemExit(f"CRITICAL: In-memory GPU smoke test failed: {e}")

print("\n=== GATELIGHT 2: REAL DATASET BATCH & ARCHITECTURE VALIDATION ===")
architectures = ("siamese_unet", "vit", "swin")

# Test LEVIR-CD real batch if available
if dataset_status["levir_ok"]:
    print("Testing real LEVIR-CD batch against all 3 architectures...")
    levir_ds = LEVIRCDDataset(ACTIVE_LEVIR_DIR, split="train", patch_size=256, stride=256)
    levir_loader = DataLoader(levir_ds, batch_size=2, shuffle=False)
    levir_batch = next(iter(levir_loader))
    b_t1 = levir_batch["before"].to(device)
    b_t2 = levir_batch["after"].to(device)
    b_mask = levir_batch["mask"].to(device)
    assert b_t1.shape == (2, 3, 256, 256), f"LEVIR-CD unexpected batch shape: {b_t1.shape}"

    for arch in architectures:
        m = get_model(arch, in_channels=3, img_size=256, pretrained=False).to(device)
        m.eval()
        with torch.no_grad():
            out = m(b_t1, b_t2)
            assert out.shape == (2, 1, 256, 256), f"{arch} failed on LEVIR-CD: output shape {out.shape}"
        print(f"   {arch:<15} on LEVIR-CD real batch: OK")

# Test OSCD real batch if available
if dataset_status["oscd_ok"]:
    print("Testing real OSCD Sentinel-2 batch against all 3 architectures...")
    oscd_ds = OSCDDataset(ACTIVE_OSCD_DIR, split="train", bands=("B04", "B03", "B02"), patch_size=256, stride=256)
    oscd_loader = DataLoader(oscd_ds, batch_size=2, shuffle=False)
    oscd_batch = next(iter(oscd_loader))
    o_t1 = oscd_batch["before"].to(device)
    o_t2 = oscd_batch["after"].to(device)
    o_mask = oscd_batch["mask"].to(device)
    assert o_t1.shape == (2, 3, 256, 256), f"OSCD unexpected batch shape: {o_t1.shape}"

    for arch in architectures:
        m = get_model(arch, in_channels=3, img_size=256, pretrained=False).to(device)
        m.eval()
        with torch.no_grad():
            out = m(o_t1, o_t2)
            assert out.shape == (2, 1, 256, 256), f"{arch} failed on OSCD: output shape {out.shape}"
        print(f"   {arch:<15} on OSCD real batch: OK")

print("=== ALL PRE-FLIGHT VERIFICATION GATES PASSED CLEANLY ===")


## Step 8: Systematic Model Training Pipeline

### Controlled Research Configuration:
Every architecture is trained with strictly identical optimization parameters:
- **Optimizer**: `AdamW` (`weight_decay=1e-4`)
- **Loss Function**: Combined Binary Cross-Entropy + Dice Loss (`bce_dice`)
- **Learning Rate**: $3 \times 10^{-4}$ with Cosine Annealing (`--scheduler cosine`)
- **Batch Size**: 8 (starting baseline; change to 4 or 2 if VRAM pressure occurs)
- **Patch Resolution**: $256 \times 256$ pixels
- **Random Seed**: `42` across all Python, NumPy, and PyTorch RNGs

### Checkpoint Persistence & Anti-Stale Safeguards:
- **Periodic Drive Sync**: `--backup-dir` copies checkpoints to Google Drive at the end of **every epoch**.
- **`FORCE_RETRAIN = False` (Default)**: Completed experiments are skipped. Interrupted experiments resume from `latest.pt`.
- **`FORCE_RETRAIN = True`**: Safely segregates pre-existing checkpoints into `checkpoints/archived/`. Old checkpoints are never evaluated if a new training run fails.


In [ ]:
import subprocess
import json
import time
import torch

# Toggle FORCE_RETRAIN:
# - False: Skip completed runs; resume interrupted runs from latest.pt.
# - True: Segregate old checkpoints into archived/ and train fresh from epoch 1.
FORCE_RETRAIN = False

EXPERIMENT_MATRIX = [
    # --- LEVIR-CD EXPERIMENTS ---
    {
        "id": "exp1_levir_unet",
        "model": "siamese_unet",
        "dataset": "LEVIR-CD",
        "data_dir": str(ACTIVE_LEVIR_DIR),
        "epochs": 20,
        "batch_size": 8,
        "bands": None,
        "enabled": dataset_status["levir_ok"],
    },
    {
        "id": "exp2_levir_vit",
        "model": "vit",
        "dataset": "LEVIR-CD",
        "data_dir": str(ACTIVE_LEVIR_DIR),
        "epochs": 20,
        "batch_size": 8,
        "bands": None,
        "enabled": dataset_status["levir_ok"],
    },
    {
        "id": "exp3_levir_swin",
        "model": "swin",
        "dataset": "LEVIR-CD",
        "data_dir": str(ACTIVE_LEVIR_DIR),
        "epochs": 20,
        "batch_size": 8,
        "bands": None,
        "enabled": dataset_status["levir_ok"],
    },
    # --- OSCD EXPERIMENTS ---
    {
        "id": "exp4_oscd_unet",
        "model": "siamese_unet",
        "dataset": "OSCD",
        "data_dir": str(ACTIVE_OSCD_DIR),
        "epochs": 25,
        "batch_size": 8,
        "bands": ["B04", "B03", "B02"],
        "enabled": dataset_status["oscd_ok"],
    },
    {
        "id": "exp5_oscd_vit",
        "model": "vit",
        "dataset": "OSCD",
        "data_dir": str(ACTIVE_OSCD_DIR),
        "epochs": 25,
        "batch_size": 8,
        "bands": ["B04", "B03", "B02"],
        "enabled": dataset_status["oscd_ok"],
    },
    {
        "id": "exp6_oscd_swin",
        "model": "swin",
        "dataset": "OSCD",
        "data_dir": str(ACTIVE_OSCD_DIR),
        "epochs": 25,
        "batch_size": 8,
        "bands": ["B04", "B03", "B02"],
        "enabled": dataset_status["oscd_ok"],
    },
]

manifest_path = DRIVE_OUTPUT_DIR / "experiment_manifest.json"
manifest = {}
if manifest_path.exists():
    try:
        manifest = json.loads(manifest_path.read_text())
    except Exception:
        manifest = {}

print(f"Loaded existing manifest with {len(manifest)} recorded runs.")

for idx, exp in enumerate(EXPERIMENT_MATRIX, 1):
    exp_id = exp["id"]
    model_name = exp["model"]
    dataset_name = exp["dataset"]
    safe_ds = dataset_name.lower().replace(" ", "_")
    
    local_ckpt_dir = Path("models/checkpoints")
    local_best = local_ckpt_dir / f"{model_name}_{safe_ds}_best.pt"
    local_latest = local_ckpt_dir / f"{model_name}_{safe_ds}_latest.pt"
    
    best_ckpt_drive = DRIVE_CHECKPOINTS_DIR / f"{model_name}_{safe_ds}_best.pt"
    latest_ckpt_drive = DRIVE_CHECKPOINTS_DIR / f"{model_name}_{safe_ds}_latest.pt"

    print(f"\n=======================================================")
    print(f"[{idx}/6] Experiment: {exp_id} ({model_name} on {dataset_name})")
    print(f"=======================================================")

    if not exp["enabled"]:
        print(f"  SKIPPED: Dataset for {dataset_name} is marked disabled/unavailable.")
        continue

    # Unique run identifier for this specific training attempt
    run_timestamp = int(time.time())
    run_id = f"{exp_id}_{run_timestamp}"

    # --- HANDLE FORCE_RETRAIN VS RESUME ---
    if FORCE_RETRAIN:
        # Segregate previous Drive checkpoint into archived/ so it cannot be mistaken for a fresh output
        if best_ckpt_drive.exists():
            archive_p = DRIVE_ARCHIVE_DIR / f"{model_name}_{safe_ds}_archived_{run_timestamp}.pt"
            shutil.copy2(best_ckpt_drive, archive_p)
            best_ckpt_drive.unlink()
            print(f"  [FORCE_RETRAIN] Safely archived old Drive checkpoint to: {archive_p.name}")
        if latest_ckpt_drive.exists():
            latest_ckpt_drive.unlink()
        if local_best.exists():
            local_best.unlink()
        if local_latest.exists():
            local_latest.unlink()
        print(f"  [FORCE_RETRAIN] Starting fresh training from epoch 1 (run_id: {run_id}).")
    else:
        # Check if already completed and verified
        manifest_entry = manifest.get(exp_id, {})
        if best_ckpt_drive.exists() and manifest_entry.get("status") == "COMPLETED":
            try:
                meta = torch.load(best_ckpt_drive, map_location="cpu", weights_only=False)
                if meta.get("saved_at") == manifest_entry.get("checkpoint_saved_at"):
                    print(f"  SKIPPED: Already completed with verified checkpoint: {best_ckpt_drive.name}")
                    continue
            except Exception:
                pass  # Fall through to retrain/resume if checkpoint cannot be verified

    # Mark as RUNNING in the manifest
    manifest[exp_id] = {
        "run_id": run_id,
        "status": "RUNNING",
        "model": model_name,
        "dataset": dataset_name,
        "checkpoint": None,
        "checkpoint_saved_at": None,
        "started_at": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
    }
    manifest_path.write_text(json.dumps(manifest, indent=2))

    cmd = [
        sys.executable,
        "scripts/train_model.py",
        "--model", model_name,
        "--dataset", dataset_name,
        "--data-dir", exp["data_dir"],
        "--epochs", str(exp["epochs"]),
        "--batch-size", str(exp["batch_size"]),
        "--patch-size", "256",
        "--lr", "3e-4",
        "--loss", "bce_dice",
        "--optimizer", "adamw",
        "--scheduler", "cosine",
        "--device", "cuda",
        "--seed", "42",
        "--checkpoint-dir", "models/checkpoints",
        "--experiments-dir", "experiments",
        "--backup-dir", str(DRIVE_CHECKPOINTS_DIR),
        "--experiment-id", run_id,
    ]

    # Resume interrupted run only when not forcing a retrain
    if not FORCE_RETRAIN and latest_ckpt_drive.exists():
        print(f"  Found previous run checkpoint! Resuming from: {latest_ckpt_drive.name}")
        cmd.extend(["--resume", str(latest_ckpt_drive)])

    if exp["bands"]:
        cmd.append("--bands")
        cmd.extend(exp["bands"])

    print(f"  Launching training command: {' '.join(cmd[:6])} ...")
    t0 = time.time()
    try:
        proc = subprocess.run(cmd, check=True)
        duration_sec = round(time.time() - t0, 2)
        
        # Verify that local_best was genuinely created during THIS run (mtime >= t0)
        if local_best.exists() and local_best.stat().st_mtime >= (t0 - 5.0):
            meta = torch.load(local_best, map_location="cpu", weights_only=False)
            shutil.copy2(local_best, best_ckpt_drive)
            
            manifest[exp_id] = {
                "run_id": run_id,
                "status": "COMPLETED",
                "model": model_name,
                "dataset": dataset_name,
                "checkpoint": str(best_ckpt_drive),
                "checkpoint_saved_at": meta.get("saved_at"),
                "epochs_completed": meta.get("epoch"),
                "best_val_f1": meta.get("best_metric"),
                "duration_sec": duration_sec,
                "completed_at": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
            }
            print(f"  Experiment {exp_id} COMPLETED in {duration_sec}s. Checkpoint synced to Drive.")
        else:
            manifest[exp_id] = {
                "run_id": run_id,
                "status": "FAILED",
                "model": model_name,
                "dataset": dataset_name,
                "error": "No freshly produced best checkpoint found after training",
                "duration_sec": duration_sec,
            }
            print(f"  WARNING: Training finished but no new best checkpoint was produced.")
    except subprocess.CalledProcessError as err:
        duration_sec = round(time.time() - t0, 2)
        manifest[exp_id] = {
            "run_id": run_id,
            "status": "FAILED",
            "model": model_name,
            "dataset": dataset_name,
            "error": f"Process exited with code {err.returncode}",
            "duration_sec": duration_sec,
        }
        print(f"  ERROR: Experiment {exp_id} failed with exit code {err.returncode}. Continuing remaining runs.")
    except Exception as exc:
        manifest[exp_id] = {
            "run_id": run_id,
            "status": "FAILED",
            "model": model_name,
            "dataset": dataset_name,
            "error": str(exc),
        }
        print(f"  ERROR: Unexpected failure in {exp_id}: {exc}. Continuing remaining runs.")

    # Save manifest snapshot to Google Drive immediately after each run
    manifest_path.write_text(json.dumps(manifest, indent=2))

print("\n Training sequence finished. Manifest saved to Google Drive.")


## Step 9: Pre-Evaluation Checkpoint Provenance Gate
Defines a strict validator that inspects both the experiment manifest and checkpoint headers before benchmarking.
If any model failed or has stale weights from an earlier session, evaluation is aborted.


In [ ]:
import json
import torch

def verify_dataset_provenance(dataset_name: str, required_models: list[str]) -> tuple[bool, list[str]]:
    manifest_p = DRIVE_OUTPUT_DIR / "experiment_manifest.json"
    if not manifest_p.exists():
        return False, ["experiment_manifest.json does not exist. Run Step 8 training first."]
    
    current_manifest = json.loads(manifest_p.read_text())
    issues = []
    
    for m in required_models:
        # Find matching manifest record
        entry = None
        for k, v in current_manifest.items():
            if v.get("model") == m and v.get("dataset") == dataset_name:
                entry = v
                break
        
        if not entry:
            issues.append(f"{m}: Never executed / missing from manifest")
            continue
        
        status = entry.get("status")
        if status != "COMPLETED":
            issues.append(f"{m}: Status is {status} (Error: {entry.get('error')})")
            continue
            
        ckpt_str = entry.get("checkpoint")
        if not ckpt_str or not Path(ckpt_str).is_file():
            issues.append(f"{m}: Checkpoint file missing on Drive: {ckpt_str}")
            continue
            
        # Header verification to prevent stale checkpoint evaluation
        try:
            meta = torch.load(ckpt_str, map_location="cpu", weights_only=False)
            recorded_saved_at = entry.get("checkpoint_saved_at")
            actual_saved_at = meta.get("saved_at")
            if recorded_saved_at != actual_saved_at:
                issues.append(
                    f"{m}: Stale checkpoint detected! File saved_at ({actual_saved_at}) != manifest ({recorded_saved_at})"
                )
        except Exception as exc:
            issues.append(f"{m}: Could not read checkpoint header: {exc}")
            
    return len(issues) == 0, issues


### Benchmark LEVIR-CD on Dedicated Test Split
Evaluates the three models on the 2,048 test patches of LEVIR-CD under identical conditions:
- **Metrics**: Precision, Recall, F1-Score, IoU, Overall Accuracy, Kappa
- **Visuals**: Pixel-level Confusion Matrices & Qualitative Prediction Overlays
- **Outputs**: Saved directly to Google Drive (`DRIVE_EXPERIMENTS_DIR` and `DRIVE_VIZ_DIR`)


In [ ]:
if dataset_status["levir_ok"]:
    print("=== Checking Provenance for LEVIR-CD Checkpoints ===")
    required_models = ["siamese_unet", "vit", "swin"]
    is_valid, issues = verify_dataset_provenance("LEVIR-CD", required_models)
    
    if not is_valid:
        print(" ERROR: LEVIR-CD benchmark aborted due to checkpoint provenance failures:")
        for issue in issues:
            print(f"   • {issue}")
        print("Stale or unverified checkpoints cannot pass this gate. Please resolve training failures first.")
    else:
        print(" All 3 model checkpoints verified against manifest! Running LEVIR-CD benchmark on TEST split...")
        subprocess.run([
            sys.executable,
            "scripts/evaluate_model.py",
            "--dataset", "LEVIR-CD",
            "--data-dir", str(ACTIVE_LEVIR_DIR),
            "--split", "test",
            "--patch-size", "256",
            "--batch-size", "8",
            "--threshold", "0.5",
            "--device", "cuda",
            "--checkpoint-dir", str(DRIVE_CHECKPOINTS_DIR),
            "--experiments-dir", str(DRIVE_EXPERIMENTS_DIR),
            "--viz-dir", str(DRIVE_VIZ_DIR),
            "--save-visuals",
            "--visual-samples", "6",
        ], check=True)


### Benchmark OSCD on Dedicated Test Split
Evaluates the three models on the 10 official test cities (69 patches) of OSCD under identical conditions.


In [ ]:
if dataset_status["oscd_ok"]:
    print("=== Checking Provenance for OSCD Checkpoints ===")
    required_models = ["siamese_unet", "vit", "swin"]
    is_valid, issues = verify_dataset_provenance("OSCD", required_models)
    
    if not is_valid:
        print(" ERROR: OSCD benchmark aborted due to checkpoint provenance failures:")
        for issue in issues:
            print(f"   • {issue}")
        print("Stale or unverified checkpoints cannot pass this gate. Please resolve training failures first.")
    else:
        print(" All 3 model checkpoints verified against manifest! Running OSCD benchmark on TEST split...")
        subprocess.run([
            sys.executable,
            "scripts/evaluate_model.py",
            "--dataset", "OSCD",
            "--data-dir", str(ACTIVE_OSCD_DIR),
            "--split", "test",
            "--patch-size", "256",
            "--batch-size", "8",
            "--threshold", "0.5",
            "--device", "cuda",
            "--bands", "B04", "B03", "B02",
            "--checkpoint-dir", str(DRIVE_CHECKPOINTS_DIR),
            "--experiments-dir", str(DRIVE_EXPERIMENTS_DIR),
            "--viz-dir", str(DRIVE_VIZ_DIR),
            "--save-visuals",
            "--visual-samples", "6",
        ], check=True)


## Step 10: Render Model Comparison Charts & Confusion Matrices


In [ ]:
import glob
from IPython.display import Image, display

# 1. Comparison bar charts from Drive
charts = sorted(glob.glob(str(DRIVE_VIZ_DIR / "*_comparison.png")))
if charts:
    print(f"Found {len(charts)} comparison charts on Drive:")
    for c in charts:
        print(f"Displaying: {c}")
        display(Image(filename=c))
else:
    print("No comparison charts found in visualizations folder.")

# 2. Confusion Matrices from Drive
cms = sorted(glob.glob(str(DRIVE_VIZ_DIR / "*_cm.png")))
if cms:
    print(f"\nFound {len(cms)} confusion matrices on Drive:")
    for cm in cms:
        display(Image(filename=cm))


## Step 11: Final Google Drive Backup Audit & Verification
Audits the persistent Google Drive directories to confirm that all trained checkpoints, evaluation summaries, confusion matrices, and prediction samples exist with valid non-zero file sizes.


In [ ]:
import os
import time

print("=== AUDIT: Verified Persistent Files on Google Drive ===")

def audit_dir(name: str, path: Path):
    print(f"\n[{name}] -> {path}")
    if not path.is_dir():
        print("  Directory does not exist!")
        return
    files = sorted(path.rglob("*.*"))
    if not files:
        print("  (Empty directory)")
        return
    for f in files:
        if f.is_file():
            size_mb = f.stat().st_size / (1024 * 1024)
            rel_name = str(f.relative_to(path))
            mtime = time.strftime("%Y-%m-%d %H:%M:%S", time.localtime(f.stat().st_mtime))
            print(f"   {rel_name:<52} | {size_mb:>8.2f} MB | {mtime}")

audit_dir("Active Checkpoints", DRIVE_CHECKPOINTS_DIR)
audit_dir("Archived Checkpoints", DRIVE_ARCHIVE_DIR)
audit_dir("Experiments & Benchmarks", DRIVE_EXPERIMENTS_DIR)
audit_dir("Visualizations & Predictions", DRIVE_VIZ_DIR)

print("\n Audit completed. All experimental assets are safely preserved on Google Drive.")
